# IMERG data pipeline: split và chuẩn hóa

Notebook này nối hai file IMERG hiện có, loại bỏ frame trùng ở mốc `2025-01-01`, chia dữ liệu theo thời gian và tạo bản chuẩn hóa cho mô hình.

- **Train:** `2023-01-01` đến trước `2025-01-01`
- **Validation:** `2025-01-01` đến trước `2025-07-01`
- **Test:** từ `2025-07-01` đến timestamp thực tế cuối cùng đã tải

Dữ liệu sau `2025-09-30 23:30` hiện chưa có trong file 2025-2026, nên notebook không tự điền số 0 và không đưa phần thiếu vào test.

In [1]:
# Cell 2 - Cấu hình thư mục và tham số dữ liệu.
# Chạy được trên Google Colab hoặc máy local.
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/Rainfall_Nowcasting/IMERG")
else:
    PROJECT_DIR = Path.cwd().resolve()
    if not (PROJECT_DIR / "imerg_vietnam_2023_2024.nc").exists() and (PROJECT_DIR / "IMERG").is_dir():
        PROJECT_DIR = PROJECT_DIR / "IMERG"

DATA_DIR = PROJECT_DIR / "data" / "imerg_vietnam"
LOCAL_DIR = PROJECT_DIR
TRAIN_SOURCE = DATA_DIR / "imerg_vietnam_2023_2024.nc"
TEST_SOURCE = DATA_DIR / "imerg_vietnam_2025_2026.nc"

# Nếu chạy notebook ngay trong thư mục IMERG và không dùng Drive, dùng file local.
if not TRAIN_SOURCE.exists() and (LOCAL_DIR / "imerg_vietnam_2023_2024.nc").exists():
    TRAIN_SOURCE = LOCAL_DIR / "imerg_vietnam_2023_2024.nc"
if not TEST_SOURCE.exists() and (LOCAL_DIR / "imerg_vietnam_2025_2026.nc").exists():
    TEST_SOURCE = LOCAL_DIR / "imerg_vietnam_2025_2026.nc"

OUTPUT_DIR = PROJECT_DIR / "data" / "prepared"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_START = pd.Timestamp("2023-01-01 00:00:00")
VAL_START = pd.Timestamp("2025-01-01 00:00:00")
TEST_START = pd.Timestamp("2025-07-01 00:00:00")
VAL_END = TEST_START
INPUT_STEPS = 6
FORECAST_STEPS = 4
WINDOW_SIZE = INPUT_STEPS + FORECAST_STEPS
TIME_STEP = pd.Timedelta(minutes=30)

print(f"Train source: {TRAIN_SOURCE}")
print(f"2025-2026 source: {TEST_SOURCE}")
print(f"Output directory: {OUTPUT_DIR}")

Train source: H:\PBL6\Rainfall_Nowcasting\IMERG\imerg_vietnam_2023_2024.nc
2025-2026 source: H:\PBL6\Rainfall_Nowcasting\IMERG\imerg_vietnam_2025_2026.nc
Output directory: H:\PBL6\Rainfall_Nowcasting\IMERG\data\prepared


## 1. Đọc, nối và kiểm tra dữ liệu

Hai file có thể trùng frame tại `2025-01-01 00:00`. Ta nối theo thời gian, sắp xếp và giữ một bản ghi duy nhất. Không gán giá trị 0 cho khoảng thời gian chưa tải.

In [2]:
# Cell 4 - Đọc rainfall từ hai NetCDF và nối thành một chuỗi thời gian.
def load_rainfall(path):
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file dữ liệu: {path}")
    with xr.open_dataset(path) as dataset:
        variable_name = "rainfall" if "rainfall" in dataset.data_vars else list(dataset.data_vars)[0]
        data = dataset[variable_name].load()
    if "time" not in data.dims:
        raise ValueError(f"Biến rainfall phải có dimension time, nhận được: {data.dims}")
    return data.sortby("time")

rain_2023_2024 = load_rainfall(TRAIN_SOURCE)
rain_2025_2026 = load_rainfall(TEST_SOURCE)

# concat rồi groupby(time).first() loại bỏ frame trùng ở ranh giới hai file.
rain = xr.concat([rain_2023_2024, rain_2025_2026], dim="time").sortby("time")
rain = rain.groupby("time").first()
actual_time = pd.DatetimeIndex(pd.to_datetime(rain.time.values))

if actual_time.has_duplicates:
    raise ValueError("Vẫn còn timestamp trùng sau khi gộp dữ liệu.")
if not np.isfinite(rain.values).any():
    raise ValueError("Không có giá trị rainfall hợp lệ.")

print(f"Combined shape: {rain.shape}")
print(f"Combined range: {actual_time.min()} -> {actual_time.max()}")
print(f"Combined frames: {len(actual_time):,}")
print(f"Duplicate timestamps after merge: {actual_time.duplicated().sum():,}")

expected_full = pd.date_range(actual_time.min(), actual_time.max(), freq=TIME_STEP)
missing_full = expected_full.difference(actual_time)
print(f"Missing timestamps inside observed range: {len(missing_full):,}")
if len(missing_full):
    print("First missing timestamps:", missing_full[:10].tolist())

Combined shape: (48192, 165, 80)
Combined range: 2023-01-01 00:00:00 -> 2025-09-30 23:30:00
Combined frames: 48,192
Duplicate timestamps after merge: 0
Missing timestamps inside observed range: 0


## 2. Chia train, validation và test

Split theo thời gian, không chia ngẫu nhiên:

- Train dùng toàn bộ 2023-2024.
- Validation dùng 6 tháng đầu 2025.
- Test dùng phần dữ liệu đã thực sự tải từ `2025-07-01` trở đi.

Nếu dữ liệu 2025-2026 chưa tải hết, test chỉ phản ánh phần quan sát hiện có.

In [3]:
# Cell 6 - Tạo ba split theo thời gian và ghi ra NetCDF riêng.
def select_period(data, start, end=None):
    if end is None:
        return data.sel(time=data.time >= np.datetime64(start))
    return data.sel(time=(data.time >= np.datetime64(start)) & (data.time < np.datetime64(end)))

train = select_period(rain, TRAIN_START, VAL_START)
validation = select_period(rain, VAL_START, VAL_END)
test = select_period(rain, TEST_START)

splits = {"train": train, "validation": validation, "test": test}
for name, data in splits.items():
    if data.sizes.get("time", 0) == 0:
        raise ValueError(f"Split {name!r} không có frame.")
    timestamps = pd.DatetimeIndex(pd.to_datetime(data.time.values))
    print(f"{name}: {len(timestamps):,} frames | {timestamps.min()} -> {timestamps.max()}")
    split_path = OUTPUT_DIR / f"imerg_{name}_raw.nc"
    data.to_netcdf(split_path)
    print(f"  Saved: {split_path}")

print("\nLưu ý: test không chứa các tháng sau timestamp cuối cùng hiện có trong file 2025-2026.")

train: 35,088 frames | 2023-01-01 00:00:00 -> 2024-12-31 23:30:00
  Saved: H:\PBL6\Rainfall_Nowcasting\IMERG\data\prepared\imerg_train_raw.nc
validation: 8,688 frames | 2025-01-01 00:00:00 -> 2025-06-30 23:30:00
  Saved: H:\PBL6\Rainfall_Nowcasting\IMERG\data\prepared\imerg_validation_raw.nc
test: 4,416 frames | 2025-07-01 00:00:00 -> 2025-09-30 23:30:00
  Saved: H:\PBL6\Rainfall_Nowcasting\IMERG\data\prepared\imerg_test_raw.nc

Lưu ý: test không chứa các tháng sau timestamp cuối cùng hiện có trong file 2025-2026.


## 3. Tạo Vietnam land mask

Grid IMERG là bounding box quanh Việt Nam, không phải toàn bộ pixel đều nằm trên đất liền. Cell này dùng polygon Natural Earth để tạo:

- `land_mask`: `1` trong polygon Việt Nam, `0` ngoài polygon.
- `W_land`: `2.5` trên đất liền Việt Nam, `1.0` trên vùng biển.

`W_land` được lưu riêng để nhóm model dùng cho Vietnam Land-Weighted L1 loss.

In [5]:
# Cell 8 - Tạo land mask và trọng số đất liền Việt Nam.
# Tự cài geopandas/shapely nếu kernel hiện tại chưa có package.
import importlib.util

missing_packages = [
    package
    for package in ("geopandas", "shapely")
    if importlib.util.find_spec(package) is None
]
if missing_packages:
    import subprocess

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing_packages,
    ])

import geopandas as gpd
import torch
from shapely.geometry import Point

NATURAL_EARTH_URL = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"
land_file = OUTPUT_DIR / "vietnam_land_mask.pt"
weight_file = OUTPUT_DIR / "vietnam_land_weight.pt"

countries = gpd.read_file(NATURAL_EARTH_URL).to_crs("EPSG:4326")
name_column = "ADMIN" if "ADMIN" in countries.columns else "NAME_EN"
vietnam = countries[countries[name_column].astype(str).str.contains("viet", case=False, na=False)]
if vietnam.empty:
    raise ValueError("Không tìm thấy polygon Việt Nam trong Natural Earth dataset.")

latitudes = np.asarray(rain.lat.values)
longitudes = np.asarray(rain.lon.values)
longitude_grid, latitude_grid = np.meshgrid(longitudes, latitudes)
points = gpd.GeoSeries(
    [Point(lon, lat) for lon, lat in zip(longitude_grid.ravel(), latitude_grid.ravel())],
    crs="EPSG:4326",
)

# covers gồm cả điểm nằm đúng trên biên polygon.
geometry = vietnam.geometry.union_all() if hasattr(vietnam.geometry, "union_all") else vietnam.geometry.unary_union
inside = points.apply(geometry.covers).to_numpy().reshape(latitude_grid.shape)
land_mask = xr.DataArray(
    inside.astype("uint8"),
    dims=("lat", "lon"),
    coords={"lat": rain.lat, "lon": rain.lon},
    name="land_mask",
)
land_weight = xr.DataArray(
    np.where(inside, 2.5, 1.0).astype("float32"),
    dims=("lat", "lon"),
    coords={"lat": rain.lat, "lon": rain.lon},
    name="W_land",
)

# PyTorch tensors giữ đúng giao diện cho loss: (lat, lon).
torch.save(torch.from_numpy(land_mask.values), land_file)
torch.save(torch.from_numpy(land_weight.values), weight_file)
land_mask.to_netcdf(OUTPUT_DIR / "vietnam_land_mask.nc")
land_weight.to_netcdf(OUTPUT_DIR / "vietnam_land_weight.nc")

print(f"Vietnam land pixels: {int(inside.sum()):,}/{inside.size:,} ({inside.mean():.2%})")
print(f"Saved binary mask: {land_file}")
print(f"Saved loss weights: {weight_file}")

Vietnam land pixels: 2,802/13,200 (21.23%)
Saved binary mask: H:\PBL6\Rainfall_Nowcasting\IMERG\data\prepared\vietnam_land_mask.pt
Saved loss weights: H:\PBL6\Rainfall_Nowcasting\IMERG\data\prepared\vietnam_land_weight.pt


## 3. Tạo sliding window cho nowcasting

Theo interface trong `Task_Division_Phase1.md`, mỗi sample có:

- Input: 6 frame liên tiếp, shape `(6, 165, 80, 1)`.
- Target: 4 frame kế tiếp, shape `(4, 165, 80, 1)`.

Chỉ giữ sample khi 10 timestamp liên tiếp cách nhau 30 phút và nằm hoàn toàn trong cùng một split.

In [6]:
# Cell 8 - Kiểm tra window hợp lệ và lưu danh sách index.
def valid_window_starts(data):
    timestamps = pd.DatetimeIndex(pd.to_datetime(data.time.values))
    valid = []
    for start in range(0, len(timestamps) - WINDOW_SIZE + 1):
        window = timestamps[start : start + WINDOW_SIZE]
        if (window[1:] - window[:-1] == TIME_STEP).all():
            frame_values = data.isel(time=slice(start, start + WINDOW_SIZE)).values
            if np.isfinite(frame_values).all():
                valid.append(start)
    return np.asarray(valid, dtype=np.int64)

window_indices = {}
for name, data in splits.items():
    indices = valid_window_starts(data)
    window_indices[name] = indices
    np.save(OUTPUT_DIR / f"{name}_window_starts.npy", indices)
    print(f"{name}: {len(indices):,} valid windows")

print("Các window bị loại nếu có gap timestamp, NaN hoặc vượt ranh giới split.")

train: 35,079 valid windows
validation: 8,679 valid windows
test: 4,407 valid windows
Các window bị loại nếu có gap timestamp, NaN hoặc vượt ranh giới split.


## 4. Chuẩn hóa rainfall

Chuẩn hóa theo hai bước:

1. `log1p(x) = log(1 + x)` để giảm lệch do nhiều pixel không mưa và một số pixel mưa lớn.
2. Z-score dùng `train_mean` và `train_std` tính **chỉ trên train**.

Validation và test không được dùng để tính thống kê, tránh data leakage. Có thể khôi phục đơn vị gốc bằng `expm1(normalized * train_std + train_mean)`.

In [7]:
# Cell 10 - Tính thống kê train, chuẩn hóa ba split và lưu kết quả.
# Không tính mean/std riêng cho validation hoặc test.
train_values = train.values.astype("float32")
if np.nanmin(train_values) < 0:
    raise ValueError("Rainfall có giá trị âm, cần xử lý missing/fill trước khi chuẩn hóa.")

train_log = np.log1p(train_values)
train_mean = float(np.nanmean(train_log))
train_std = float(np.nanstd(train_log))
if not np.isfinite(train_std) or train_std == 0:
    raise ValueError(f"train_std không hợp lệ: {train_std}")

normalization_stats = {
    "transform": "log1p_then_train_zscore",
    "train_log_mean": train_mean,
    "train_log_std": train_std,
    "input_units": "mm/30 min",
    "formula": "z = (log1p(rainfall) - train_log_mean) / train_log_std",
    "inverse_formula": "rainfall = expm1(z * train_log_std + train_log_mean)",
}

for name, data in splits.items():
    values = data.values.astype("float32")
    if np.nanmin(values) < 0:
        raise ValueError(f"Split {name} có giá trị âm.")
    normalized = (np.log1p(values) - train_mean) / train_std
    normalized_data = xr.DataArray(
        normalized.astype("float32"),
        dims=data.dims,
        coords=data.coords,
        name="rainfall_normalized",
        attrs={
            "units": "train-standardized log1p(mm/30 min)",
            "train_log_mean": train_mean,
            "train_log_std": train_std,
        },
    )
    output_path = OUTPUT_DIR / f"imerg_{name}_normalized.nc"
    normalized_data.to_netcdf(output_path)
    print(f"{name} normalized: {output_path}")

metadata = {
    "source_files": [str(TRAIN_SOURCE), str(TEST_SOURCE)],
    "observed_start": actual_time.min().isoformat(),
    "observed_end": actual_time.max().isoformat(),
    "missing_after_observed_end_is_not_filled": True,
    "split_boundaries": {
        "train": [TRAIN_START.isoformat(), VAL_START.isoformat()],
        "validation": [VAL_START.isoformat(), VAL_END.isoformat()],
        "test": [TEST_START.isoformat(), actual_time.max().isoformat()],
    },
    "input_steps": INPUT_STEPS,
    "forecast_steps": FORECAST_STEPS,
    "time_step_minutes": 30,
    "normalization": normalization_stats,
    "frame_counts": {name: int(data.sizes["time"]) for name, data in splits.items()},
    "valid_window_counts": {name: int(len(indices)) for name, indices in window_indices.items()},
}
(OUTPUT_DIR / "dataset_metadata.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
print(f"Train log mean: {train_mean:.8f}")
print(f"Train log std: {train_std:.8f}")
print(f"Metadata saved: {OUTPUT_DIR / 'dataset_metadata.json'}")

train normalized: H:\PBL6\Rainfall_Nowcasting\IMERG\data\prepared\imerg_train_normalized.nc
validation normalized: H:\PBL6\Rainfall_Nowcasting\IMERG\data\prepared\imerg_validation_normalized.nc
test normalized: H:\PBL6\Rainfall_Nowcasting\IMERG\data\prepared\imerg_test_normalized.nc
Train log mean: 0.06256736
Train log std: 0.22905120
Metadata saved: H:\PBL6\Rainfall_Nowcasting\IMERG\data\prepared\dataset_metadata.json
